# IND320, Assignment 2

## By: Ole Andreas Grøtting
Github: [IND320 - OLEGRO99](https://github.com/olegro99/IND320-OLEGRO99.git)\
Streamlit: https://ind320-olegro99-grxpvhtmnnumfeeuqe8u7q.streamlit.app/

# Imports

In [1]:
%pip install requests
%pip install plotly
%pip install python-dotenv
%pip install python-dotenv entsoe-py
%pip install cassandra-driver
%pip install pyarrow
%pip install --upgrade pandas pyarrow
%pip install pymongo
%pip install pymongo python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# imports

In [1]:
import requests
import pandas as pd
import plotly.express as px
import os
from dotenv import load_dotenv
from entsoe import EntsoePandasClient
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, min, max, when, avg
from cassandra.cluster import Cluster
from pymongo import MongoClient


# connecting to spark and cassandra

In [2]:
# Configure Java and Hadoop before starting Spark
os.environ["JAVA_HOME"] = (
    "C:/Program Files/Eclipse Adoptium/"
    "jdk-17.0.20.101-hotspot"
)
os.environ["PYSPARK_HADOOP_VERSION"] = "without"
os.environ["HADOOP_HOME"] = (
    r"C:\Users\oleag\Desktop\skole\IND320\IND320-OLEGRO99\hadoop-3.3.1"
)
os.environ["hadoop.home.dir"] = os.environ["HADOOP_HOME"]

print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("PYSPARK_HADOOP_VERSION:", os.environ["PYSPARK_HADOOP_VERSION"])
print("HADOOP_HOME:", os.environ["HADOOP_HOME"])

JAVA_HOME: C:/Program Files/Eclipse Adoptium/jdk-17.0.20.101-hotspot
PYSPARK_HADOOP_VERSION: without
HADOOP_HOME: C:\Users\oleag\Desktop\skole\IND320\IND320-OLEGRO99\hadoop-3.3.1


In [3]:
#Connecting to Cassandra
cluster = Cluster(["localhost"], port=9042)
session = cluster.connect()

print("Connected to Cassandra!")

Connected to Cassandra!


In [4]:
os.environ["PYSPARK_HADOOP_VERSION"] = "without"

import pyspark
print("PySpark version:", pyspark.__version__)
print("PySpark location:", pyspark.__file__)

PySpark version: 3.5.5
PySpark location: c:\Users\oleag\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark\__init__.py


In [5]:
from pyspark.sql import SparkSession

# Create Spark with the Cassandra connector
spark = (
    SparkSession.builder
    .appName("SparkCassandraApp")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "com.datastax.spark:spark-cassandra-connector_2.12:3.5.1"
    )
    .config("spark.cassandra.connection.host", "localhost")
    .config("spark.cassandra.connection.port", "9042")
    .getOrCreate()
)

print("Spark version:", spark.version)

# Check whether the connector is configured
print(
    "Connector:",
    spark.sparkContext.getConf().get(
        "spark.jars.packages",
        "Not set"
    )
)

Spark version: 3.5.5
Connector: com.datastax.spark:spark-cassandra-connector_2.12:3.5.1


In [6]:
print("PySpark importert")
print("Java:", os.environ.get("JAVA_HOME"))
print("Hadoop-innstilling:", os.environ.get("PYSPARK_HADOOP_VERSION"))

PySpark importert
Java: C:/Program Files/Eclipse Adoptium/jdk-17.0.20.101-hotspot
Hadoop-innstilling: without


In [7]:
# Create Spark with the Cassandra connector
spark = (
    SparkSession.builder
    .appName("SparkCassandraApp")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "com.datastax.spark:spark-cassandra-connector_2.12:3.5.1"
    )
    .config("spark.cassandra.connection.host", "localhost")
    .config("spark.cassandra.connection.port", "9042")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Spark session ready!")

Spark version: 3.5.5
Spark session ready!


In [8]:
# Read data from Cassandra into a Spark DataFrame
df = (
    spark.read
    .format("org.apache.spark.sql.cassandra")
    .options(
        keyspace="energy",
        table="cross_border_flows"
    )
    .load()
)

df.printSchema()
df.show(5, truncate=False)

root
 |-- connection: string (nullable = false)
 |-- direction: string (nullable = false)
 |-- year: integer (nullable = false)
 |-- timestamp: timestamp (nullable = true)
 |-- flow_mw: double (nullable = true)

+----------+-----------+----+-------------------+-------+
|connection|direction  |year|timestamp          |flow_mw|
+----------+-----------+----+-------------------+-------+
|NO_2-GB   |into_norway|2025|2025-01-01 00:00:00|0.0    |
|NO_2-GB   |into_norway|2025|2025-01-01 00:15:00|0.0    |
|NO_2-GB   |into_norway|2025|2025-01-01 00:30:00|0.0    |
|NO_2-GB   |into_norway|2025|2025-01-01 00:45:00|0.0    |
|NO_2-GB   |into_norway|2025|2025-01-01 01:00:00|0.0    |
+----------+-----------+----+-------------------+-------+
only showing top 5 rows



### getting  Magasinstatistikk API's

In [9]:
base_url = "https://biapi.nve.no/magasinstatistikk/api/Magasinstatistikk/HentOffentligData"

response = requests.get(base_url)
response.raise_for_status()

data = response.json()

df = pd.DataFrame(data)

#removing rows with the value "0001-01-01T00:00:00" in the "neste_Publiseringsdato" column
df = df[df["neste_Publiseringsdato"] != "0001-01-01T00:00:00"] 

#sorting the dataframe by the "dato_Id" column in ascending order
df["dato_Id"] = pd.to_datetime(df["dato_Id"])

df = df.sort_values("dato_Id")

print(df.head(10))
print(df.dtypes)
print(df.shape)

         dato_Id omrType  omrnr  iso_aar  iso_uke  fyllingsgrad  \
10210 2019-02-17    VASS      2     2019        7      0.541280   
2960  2019-02-17      EL      4     2019        7      0.412725   
4398  2019-02-17      EL      3     2019        7      0.457661   
7163  2019-02-17      EL      2     2019        7      0.507618   
375   2019-02-17      EL      5     2019        7      0.436906   
3176  2019-02-17      EL      1     2019        7      0.363103   
14017 2019-02-17    VASS      1     2019        7      0.423739   
10596 2019-02-17    VASS      3     2019        7      0.427095   
9356  2019-02-17      NO      0     2019        7      0.456033   
14007 2019-02-24    VASS      1     2019        8      0.399306   

       kapasitet_TWh  fylling_TWh neste_Publiseringsdato  \
10210      23.237715    12.578101    2019-02-27T13:00:00   
2960       21.079208     8.699908    2019-02-27T13:00:00   
4398        8.920932     4.082766    2019-02-27T13:00:00   
7163       34.043590  

## data exploration

In [13]:
# Show the different area types and their area numbers
areas = (
    df.groupby(["omrType", "omrnr"])
      .size()
      .reset_index(name="observations")
)

print(areas)

# Count unique areas within each area type
print(df.groupby("omrType")["omrnr"].nunique())

  omrType  omrnr  observations
0      EL      1           400
1      EL      2           400
2      EL      3           400
3      EL      4           400
4      EL      5           400
5      NO      0           400
6    VASS      1           400
7    VASS      2           400
8    VASS      3           400
omrType
EL      5
NO      1
VASS    3
Name: omrnr, dtype: int64


In [14]:
%pip install --upgrade nbformat ipython

  Using cached nbformat-5.11.1-py3-none-any.whl.metadata (3.7 kB)
  Using cached ipython-9.17.1-py3-none-any.whl.metadata (4.6 kB)
  Using cached fastjsonschema-2.22.2-py3-none-any.whl.metadata (2.1 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached rpds_py-2026.9.1-cp311-cp311-win_amd64.whl.metadata (4.2 kB)
Using cached nbformat-5.11.1-py3-none-any.whl (79 kB)
Using cached ipython-9.17.1-py3-none-any.whl (639 kB)
Using cached fastjsonschema-2.22.2-py3-none-any.whl (27 kB)
Using cached jsonschema-4.26.0-py3-none-any.whl (90 kB)
Using cached attrs-26.1.0-py3-none-any.whl (67 kB)
Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl (18 kB)
Using cached referencing-0.37.0-py3-none-any.whl (26 kB)
Using cached rpds_py-2026.9.1-c


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### plotting the data

In [14]:
# Create a line plot showing reservoir filling over time
fig = px.line(
    df,
    x="dato_Id",        # Set the date column as the x-axis
    y="fylling_TWh",    # Set reservoir filling in TWh as the y-axis
    color="omrnr",      # Use different colors for the different areas
    title="Fylling i vannmagasinene per strømprisområde",
    labels={
        "dato_Id": "Dato",
        "fylling_TWh": "Fylling (TWh)",
        "omrnr": "Område"
    }
)

fig.show()

### loading ENSTO api keys

In [10]:
# Load the .env file
load_dotenv()

# Check whether the API key was loaded
api_key = os.getenv("ENTSOE_API_KEY")

if api_key:
    print("ENTSO-E API key loaded successfully!")
else:
    print("API key not found. Check the .env file location.")

# Create the ENTSO-E API client
client = EntsoePandasClient(api_key=api_key)

print("ENTSO-E client created successfully!")

ENTSO-E API key loaded successfully!
ENTSO-E client created successfully!


In [11]:
# Define a one-day test period
start = pd.Timestamp("2026-10-01", tz="UTC")
end = pd.Timestamp("2026-10-02", tz="UTC")

# Query cross-border electricity flows
flows = client.query_crossborder_flows(
    country_code_from="NO_2",
    country_code_to="DK_1",
    start=start,
    end=end
)

print(flows.head())
print("Number of observations:", len(flows))

2026-10-01 02:00:00+02:00    605.97
2026-10-01 02:15:00+02:00    647.22
2026-10-01 02:30:00+02:00    653.14
2026-10-01 02:45:00+02:00    582.93
2026-10-01 03:00:00+02:00    452.13
Freq: 15min, dtype: float64
Number of observations: 96


In [12]:
# Read the existing data from Cassandra
flows_df = (
    spark.read
    .format("org.apache.spark.sql.cassandra")
    .options(
        keyspace="energy",
        table="cross_border_flows"
    )
    .load()
)

# Display the schema and a few rows
flows_df.printSchema()
flows_df.show(5, truncate=False)

root
 |-- connection: string (nullable = false)
 |-- direction: string (nullable = false)
 |-- year: integer (nullable = false)
 |-- timestamp: timestamp (nullable = true)
 |-- flow_mw: double (nullable = true)

+----------+-------------+----+-------------------+-------+
|connection|direction    |year|timestamp          |flow_mw|
+----------+-------------+----+-------------------+-------+
|NO_2-DE_LU|out_of_norway|2024|2024-10-15 00:00:00|1314.4 |
|NO_2-DE_LU|out_of_norway|2024|2024-10-15 00:15:00|1324.48|
|NO_2-DE_LU|out_of_norway|2024|2024-10-15 00:30:00|1324.32|
|NO_2-DE_LU|out_of_norway|2024|2024-10-15 00:45:00|1331.2 |
|NO_2-DE_LU|out_of_norway|2024|2024-10-15 01:00:00|1357.76|
+----------+-------------+----+-------------------+-------+
only showing top 5 rows



In [13]:
# Connect to the ENTSO-E API
client = EntsoePandasClient(
    api_key=os.environ["ENTSOE_API_KEY"]
)

# Define the two-year period in UTC
start = pd.Timestamp("2024-10-15", tz="UTC")
end = pd.Timestamp("2026-10-15", tz="UTC")

# Query physical cross-border electricity flows
flows = client.query_crossborder_flows(
    country_code_from="NO_2",
    country_code_to="DK_1",
    start=start,
    end=end
)

# Convert the returned time series into a DataFrame
flows_df = flows.rename("flow_mw").reset_index()

# Display the retrieved data
print(flows_df.head())
print(flows_df.shape)

                      index  flow_mw
0 2024-10-15 02:00:00+02:00  1386.84
1 2024-10-15 03:00:00+02:00  1386.69
2 2024-10-15 04:00:00+02:00  1386.92
3 2024-10-15 05:00:00+02:00  1387.20
4 2024-10-15 06:00:00+02:00  1387.27
(57480, 2)


# data cleaning

In [14]:
start = pd.Timestamp("2024-10-15", tz="Europe/Brussels")
end = pd.Timestamp("2026-10-15", tz="Europe/Brussels")

print("Start:", start)
print("End:", end)

Start: 2024-10-15 00:00:00+02:00
End: 2026-10-15 00:00:00+02:00


## retriving data in and out off norway

In [15]:
# Norway to Denmark
flows_no_dk = client.query_crossborder_flows(
    country_code_from="NO_2",
    country_code_to="DK_1",
    start=start,
    end=end
)

# Denmark to Norway
flows_dk_no = client.query_crossborder_flows(
    country_code_from="DK_1",
    country_code_to="NO_2",
    start=start,
    end=end
)

print("Norway to Denmark:", len(flows_no_dk))
print("Denmark to Norway:", len(flows_dk_no))

Norway to Denmark: 57474
Denmark to Norway: 57474


In [16]:
# Define connections between Norwegian and neighbouring bidding zones
connections = [
    ("NO_1", "SE_3"),
    ("NO_2", "DK_1"),
    ("NO_2", "NL"),
    ("NO_2", "DE_LU"),
    ("NO_2", "GB"),
    ("NO_3", "SE_2"),
    ("NO_4", "SE_1"),
    ("NO_4", "SE_2"),
    ("NO_4", "FI"),
]

# Split the two-year period into two smaller requests
periods = [
    (pd.Timestamp("2024-10-15", tz="UTC"),
     pd.Timestamp("2025-10-15", tz="UTC")),
    (pd.Timestamp("2025-10-15", tz="UTC"),
     pd.Timestamp("2026-10-15", tz="UTC")),
]

all_flows = []

for no_zone, foreign_zone in connections:
    for start_period, end_period in periods:
        # Query both directions for this connection
        for source, target, direction in [
            (no_zone, foreign_zone, "out_of_norway"),
            (foreign_zone, no_zone, "into_norway"),
        ]:
            try:
                series = client.query_crossborder_flows(
                    country_code_from=source,
                    country_code_to=target,
                    start=start_period,
                    end=end_period
                )

                # Convert the time series to a DataFrame
                temp = series.rename("flow_mw").rename_axis(
                    "timestamp"
                ).reset_index()

                temp["connection"] = f"{no_zone}-{foreign_zone}"
                temp["direction"] = direction

                all_flows.append(temp)

                print(
                    f"OK: {no_zone}-{foreign_zone}, "
                    f"{direction}, {start_period.year}"
                )

            except Exception as e:
                print(
                    f"FAILED: {no_zone}-{foreign_zone}, "
                    f"{direction}, {start_period.year}: {e}"
                )

# Combine all successful API responses
if not all_flows:
    raise ValueError("No ENTSO-E data was retrieved.")

flows_df = pd.concat(all_flows, ignore_index=True)

# Remove duplicate records and sort chronologically
flows_df = flows_df.drop_duplicates(
    subset=["timestamp", "connection", "direction"]
)
flows_df = flows_df.sort_values(
    ["timestamp", "connection", "direction"]
).reset_index(drop=True)

print(flows_df.head())
print("Rows and columns:", flows_df.shape)
print(flows_df.groupby(["connection", "direction"]).size())

OK: NO_1-SE_3, out_of_norway, 2024
OK: NO_1-SE_3, into_norway, 2024
OK: NO_1-SE_3, out_of_norway, 2025
OK: NO_1-SE_3, into_norway, 2025
OK: NO_2-DK_1, out_of_norway, 2024
OK: NO_2-DK_1, into_norway, 2024
OK: NO_2-DK_1, out_of_norway, 2025
OK: NO_2-DK_1, into_norway, 2025
OK: NO_2-NL, out_of_norway, 2024
OK: NO_2-NL, into_norway, 2024
OK: NO_2-NL, out_of_norway, 2025
OK: NO_2-NL, into_norway, 2025
OK: NO_2-DE_LU, out_of_norway, 2024
OK: NO_2-DE_LU, into_norway, 2024
OK: NO_2-DE_LU, out_of_norway, 2025
OK: NO_2-DE_LU, into_norway, 2025
OK: NO_2-GB, out_of_norway, 2024
OK: NO_2-GB, into_norway, 2024
OK: NO_2-GB, out_of_norway, 2025
OK: NO_2-GB, into_norway, 2025
OK: NO_3-SE_2, out_of_norway, 2024
OK: NO_3-SE_2, into_norway, 2024
OK: NO_3-SE_2, out_of_norway, 2025
OK: NO_3-SE_2, into_norway, 2025
OK: NO_4-SE_1, out_of_norway, 2024
OK: NO_4-SE_1, into_norway, 2024
OK: NO_4-SE_1, out_of_norway, 2025
OK: NO_4-SE_1, into_norway, 2025
OK: NO_4-SE_2, out_of_norway, 2024
OK: NO_4-SE_2, into_norwa

In [17]:
# Check data types and missing values
print(flows_df.dtypes)
print(flows_df.isna().sum())

# Keep rows with valid timestamps and flow values
flows_df = flows_df.dropna(
    subset=["timestamp", "flow_mw", "connection", "direction"]
)

# Make timestamps timezone-naive in UTC for Spark compatibility
flows_df["timestamp"] = (
    pd.to_datetime(flows_df["timestamp"], utc=True)
    .dt.tz_localize(None)
)

# Inspect the cleaned dataset
flows_df.head()

timestamp      object
flow_mw       float64
connection        str
direction         str
dtype: object
timestamp     0
flow_mw       0
connection    0
direction     0
dtype: int64


,timestamp,flow_mw,connection,direction
0,2024-10-15,1034.0625,NO_1-SE_3,into_norway
1,2024-10-15,0.0000,NO_1-SE_3,out_of_norway
2,2024-10-15,0.0000,NO_2-DE_LU,into_norway
3,2024-10-15,1314.4000,NO_2-DE_LU,out_of_norway
4,2024-10-15,0.0000,NO_2-DK_1,into_norway


In [15]:
%pip install "pyspark==3.5.5"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Connect to Cassandra and Extract Data

In [18]:
# Les cross_border_flows fra Cassandra med Spark
flows_df = (
    spark.read
    .format("org.apache.spark.sql.cassandra")
    .options(
        keyspace="energy",
        table="cross_border_flows"
    )
    .load()
)

# Vis kolonnene og noen rader
flows_df.printSchema()
flows_df.show(5, truncate=False)

root
 |-- connection: string (nullable = false)
 |-- direction: string (nullable = false)
 |-- year: integer (nullable = false)
 |-- timestamp: timestamp (nullable = true)
 |-- flow_mw: double (nullable = true)

+----------+-----------+----+-------------------+-----------+
|connection|direction  |year|timestamp          |flow_mw    |
+----------+-----------+----+-------------------+-----------+
|NO_2-DK_1 |into_norway|2026|2026-01-01 00:00:00|1422.758098|
|NO_2-DK_1 |into_norway|2026|2026-01-01 00:15:00|1583.896565|
|NO_2-DK_1 |into_norway|2026|2026-01-01 00:30:00|1604.639679|
|NO_2-DK_1 |into_norway|2026|2026-01-01 00:45:00|1604.658926|
|NO_2-DK_1 |into_norway|2026|2026-01-01 01:00:00|1604.429119|
+----------+-----------+----+-------------------+-----------+
only showing top 5 rows



## Data Exploration

In [19]:
#connect to Cassandra
cluster = Cluster(["localhost"], port=9042)
session = cluster.connect()

print("Keyspaces i Cassandra:")
print(sorted(cluster.metadata.keyspaces.keys()))

Keyspaces i Cassandra:
['energy', 'system', 'system_auth', 'system_distributed', 'system_schema', 'system_traces', 'system_views', 'system_virtual_schema']


In [20]:
# Retrieve the table list directly from Cassandra
rows = session.execute(
    "SELECT table_name FROM system_schema.tables "
    "WHERE keyspace_name = 'energy'"
)

print("Tabeller i energy:")
for row in rows:
    print(row.table_name)

Tabeller i energy:
cross_border_flows


In [21]:
# Count observations for each year and direction
flows_df.groupBy("year", "direction") \
    .agg(count("*").alias("number_of_observations")) \
    .orderBy("year", "direction") \
    .show(truncate=False)

# Display all unique connections
flows_df.select("connection") \
    .distinct() \
    .orderBy("connection") \
    .show(truncate=False)

# Check the time range covered by the data
flows_df.select(
    min("timestamp").alias("first_timestamp"),
    max("timestamp").alias("last_timestamp")
).show(truncate=False)

+----+-------------+----------------------+
|year|direction    |number_of_observations|
+----+-------------+----------------------+
|2024|into_norway  |39312                 |
|2024|out_of_norway|39312                 |
|2025|into_norway  |230307                |
|2025|out_of_norway|230307                |
|2026|into_norway  |247932                |
|2026|out_of_norway|247932                |
+----+-------------+----------------------+

+----------+
|connection|
+----------+
|NO_1-SE_3 |
|NO_2-DE_LU|
|NO_2-DK_1 |
|NO_2-GB   |
|NO_2-NL   |
|NO_3-SE_2 |
|NO_4-FI   |
|NO_4-SE_1 |
|NO_4-SE_2 |
+----------+

+-------------------+-------------------+
|first_timestamp    |last_timestamp     |
+-------------------+-------------------+
|2024-10-15 00:00:00|2026-10-14 23:45:00|
+-------------------+-------------------+



In [22]:
# Count missing values in each column
flows_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in flows_df.columns
]).show()

# Count duplicate rows
total_rows = flows_df.count()
distinct_rows = flows_df.distinct().count()

print("Total rows:", total_rows)
print("Distinct rows:", distinct_rows)
print("Duplicate rows:", total_rows - distinct_rows)

+----------+---------+----+---------+-------+
|connection|direction|year|timestamp|flow_mw|
+----------+---------+----+---------+-------+
|         0|        0|   0|        0|      0|
+----------+---------+----+---------+-------+

Total rows: 1035102
Distinct rows: 1035102
Duplicate rows: 0


In [23]:
from pyspark.sql.functions import col, sum as spark_sum

# Select one year and one connection
pie_df = flows_df.filter(
    (col("year") == 2025) &
    (col("connection") == "NO_2-NL")
)

# Calculate total energy flow in MWh for each direction
pie_summary = (
    pie_df
    .groupBy("direction")
    .agg(
        (spark_sum("flow_mw") * 0.25).alias("total_mwh")
    )
    .orderBy("direction")
)

# Display the results
pie_summary.show(truncate=False)

+-------------+------------------+
|direction    |total_mwh         |
+-------------+------------------+
|into_norway  |414991.54250000144|
|out_of_norway|2791213.7150000967|
+-------------+------------------+



## pie chart

In [24]:
import plotly.express as px

# Convert the Spark DataFrame to a Pandas DataFrame
pie_pd = pie_summary.toPandas()

# Create a pie chart showing energy flow in both directions
fig = px.pie(
    pie_pd,
    names="direction",
    values="total_mwh",
    title="Total Cross-Border Electricity Flow for NO_2-NL in 2025",
    labels={
        "direction": "Flow direction",
        "total_mwh": "Total energy flow (MWh)"
    }
)

# Display the pie chart
fig.show()

I chose 2025 and the connection NO_2-NL (Norway–Netherlands) for the pie chart, as this provides a complete calendar year of data.

## line chart

In [25]:
# Select data for January 2025
january_df = flows_df.filter(
    (col("timestamp") >= "2025-01-01") &
    (col("timestamp") < "2025-02-01")
)

# Calculate average power flow per day, connection, and direction
daily_df = (
    january_df
    .groupBy(
        col("timestamp").cast("date").alias("date"),
        col("connection"),
        col("direction")
    )
    .agg(
        avg("flow_mw").alias("avg_flow_mw")
    )
    .orderBy("date")
)

# Convert the Spark DataFrame to Pandas
daily_pd = daily_df.toPandas()

# Combine connection and direction into one legend label
daily_pd["series"] = (
    daily_pd["connection"] + " - " + daily_pd["direction"]
)

# Create the line plot
fig = px.line(
    daily_pd,
    x="date",
    y="avg_flow_mw",
    color="series",
    title="Cross-Border Electricity Flows in January 2025",
    labels={
        "date": "Date",
        "avg_flow_mw": "Average Power Flow (MW)",
        "series": "Connection and Direction"
    }
)

# Display the plot
fig.show()

## Insert data into Mongo DB

In [35]:
%pip install --upgrade pymongo certifi dnspython

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
import os
from dotenv import load_dotenv

# Load the .env file from the project root
load_dotenv("../.env", override=True)

# Check whether the MongoDB URI was loaded
print("MONGO_URI found:", bool(os.getenv("MONGO_URI")))

MONGO_URI found: True


In [27]:
# Load environment variables from the project root
load_dotenv("../.env")

# Read the MongoDB connection string
mongo_uri = os.getenv("MONGO_URI")

if not mongo_uri:
    raise ValueError("MONGO_URI was not found in the .env file")

# Create the MongoDB client
client = MongoClient(
    mongo_uri,
    serverSelectionTimeoutMS=10000
)

# Test the connection
try:
    client.admin.command("ping")
    print("Connected to MongoDB successfully!")

    # Select the database and collection
    db = client["energy"]
    collection = db["cross_border_flows_spark"]

except Exception as e:
    print("MongoDB connection failed:")
    print(type(e).__name__)
    print(e)

Connected to MongoDB successfully!


In [37]:
# Insert Spark data into MongoDB in batches
from pymongo import InsertOne

batch_size = 5000
batch = []
inserted_count = 0

# Check the number of rows in the Spark DataFrame
print("Spark rows:", flows_df.count())
print("Existing MongoDB documents:", collection.count_documents({}))

# Inspect the columns before choosing a unique key
print("Spark columns:", flows_df.columns)
flows_df.show(5, truncate=False)


Spark rows: 1035102
Existing MongoDB documents: 1045902
Spark columns: ['connection', 'direction', 'year', 'timestamp', 'flow_mw']
+----------+-------------+----+-------------------+-------+
|connection|direction    |year|timestamp          |flow_mw|
+----------+-------------+----+-------------------+-------+
|NO_2-GB   |out_of_norway|2024|2024-10-15 00:00:00|0.0    |
|NO_2-GB   |out_of_norway|2024|2024-10-15 00:15:00|0.0    |
|NO_2-GB   |out_of_norway|2024|2024-10-15 00:30:00|0.0    |
|NO_2-GB   |out_of_norway|2024|2024-10-15 00:45:00|0.0    |
|NO_2-GB   |out_of_norway|2024|2024-10-15 01:00:00|0.0    |
+----------+-------------+----+-------------------+-------+
only showing top 5 rows



In [ ]:
# Check existing documents before inserting
print("Existing documents:", collection.count_documents({}))
# Check the ENTSO-E data in Spark and MongoDB
print("Spark rows:", flows_df.count())
print("MongoDB documents:", collection.count_documents({}))

Existing documents: 1045902
Spark rows: 1035102
MongoDB documents: 1045902


In [40]:
print(collection.find_one())

{'_id': ObjectId('6ad69398823fe2c4a5920724'), 'connection': 'NO_4-SE_2', 'direction': 'into_norway', 'year': 2025, 'timestamp': datetime.datetime(2025, 1, 1, 0, 0), 'flow_mw': 83.4625}


In [46]:
# Delete NVE documents from the collection
result = collection.delete_many({
    "dato_Id": {"$exists": True}
})

print("NVE documents deleted:", result.deleted_count)
print("Documents remaining:", collection.count_documents({}))

NVE documents deleted: 0
Documents remaining: 1035102


In [48]:
#checking the number of documents in the collection after deletion
print("Totalt:", collection.count_documents({}))

print(
    "ENTSO-E:",
    collection.count_documents({
        "connection": {"$exists": True},
        "direction": {"$exists": True},
        "timestamp": {"$exists": True}
    })
)

print(
    "NVE:",
    collection.count_documents({
        "dato_Id": {"$exists": True}
    })
)

Totalt: 1035102
ENTSO-E: 1035102
NVE: 0
